# ノートブック3：お手本の浮きを直して追加学習（ランタイム：A100・ハイメモリ）

ノートブック2で学習した全編の方策（`model_19999`）を出発点に、次の 2 点を変えて **4,000 反復だけ追加学習**します。

1. **お手本の浮きを解消**：終盤 30 秒（110〜140 秒）で、お手本の足が床から約 20cm 浮いていた。接地補正で「接地」とみなす足の高さを 3cm → **25cm** に広げてお手本を作り直す（GMR の計算はやり直さないので数十秒）
2. **動きのキレ**：動作の変化への罰則を半分（-0.1 → **-0.05**）、体の位置・向きの報酬の許容幅を締める（0.3 → **0.25** m、0.4 → **0.3** rad）

**事前準備**：ドライブの `pai_final/scripts/` に、手元で修正した `train_sim.py` と `render_policy.py` を上書きアップロードしておくこと。

上から順に実行すれば、お手本の作り直し → 物理チェック → 追加学習 → 評価 → 比較動画まで進みます。所要時間の目安は合計 2.5〜3 時間です。

## 1. ドライブの接続と設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
P = '/content/drive/MyDrive/pai_final'
os.environ['P'] = P

# 使うリポジトリのコミット（この版で動作を確認したソースに固定。切断後に入れ直しても同じ版になる）
PINS = {
    'GVHMR':            ('https://github.com/zju3dv/GVHMR',                  'ee960bb6e2ea2d381aa97f08e9b71ef320b624b1'),
    'GMR':              ('https://github.com/YanjieZe/GMR',                  'bb1bbe40774794fceb2a7c579a3464a28e68c844'),
    'unitree_rl_mjlab': ('https://github.com/unitreerobotics/unitree_rl_mjlab', '1425b15f73bd4095f0df53709d7c389c3eb9e790'),
    'mjlab':            ('https://github.com/mujocolab/mjlab',               '5af32e378dcb93c9e881ace83cc5a3f5d373fe60'),  # v1.2.0
}
for k, (url, rev) in PINS.items():
    os.environ[f'URL_{k}'], os.environ[f'REV_{k}'] = url, rev

# サビ後半 20 秒が、全編（伊原六花さんの動画 2分19秒）の何秒目か（音声の照合で特定済み）
SEG_START, SEG_END = 59.815, 80.78
os.environ['SEG_START'], os.environ['SEG_END'] = str(SEG_START), str(SEG_END)

# MuJoCo を画面なし（EGL）で描画するための設定
NVIDIA_ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTHONUNBUFFERED'] = '1'   # ログをすぐファイルに書き出す

# 動画・画像をノートブック上で表示する関数
from base64 import b64encode
from IPython.display import HTML, Image, display
def show(path, width=720):
    data = b64encode(open(path, 'rb').read()).decode()
    return HTML(f'<video width={width} controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

# ▼ 追加学習の設定（ノートブック3）
BASE = 'the_feels_full'                        # 元のお手本（終盤 30 秒が約 20cm 浮いている）
NAME = 'the_feels_full_g25'                    # 作り直すお手本（接地判定の高さ contact_h = 0.25 m）
BASE_RUN = '2026-10-17_08-14-52_the_feels_full'  # 読み込む学習済みの run（20,000 反復）
BASE_CKPT = 'model_19999.pt'
ITERS = 4000                                   # 追加の反復回数（3000〜5000。A100 で 4000 反復 ≒ 1.8 時間）
# 「キレ」重視の報酬（既定値: -0.1 / 0.3 / 0.4）
ACTION_RATE, BODY_POS_STD, BODY_ORI_STD = -0.05, 0.25, 0.3
for k, v in dict(BASE=BASE, NAME=NAME, BASE_RUN=BASE_RUN, BASE_CKPT=BASE_CKPT, ITERS=ITERS,
                 ACTION_RATE=ACTION_RATE, BODY_POS_STD=BODY_POS_STD, BODY_ORI_STD=BODY_ORI_STD).items():
    os.environ[k] = str(v)
os.environ['TASK'] = 'Unitree-G1-Tracking'
os.environ['START'] = '0'   # 比較動画用（全編なので 0 秒から）
L = '/content/unitree_rl_mjlab'
os.environ['RUN_SPEEDTEST'] = '0'; os.environ['RESUME'] = '0'

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls $P/outputs/motions | grep the_feels_full
!ls $P/logs/rsl_rl/g1_tracking/

## 2. unitree_rl_mjlab のインストール（5〜10分）

ノートブック2の 3-2 と同じです。ドライブの学習ログ（読み込む `model_19999.pt` を含む）がローカルに戻ります。

In [ ]:
%%bash
set -e
pip -q install uv
bash $P/scripts/pin_clone.sh $URL_unitree_rl_mjlab /content/unitree_rl_mjlab $REV_unitree_rl_mjlab
uv venv /content/venv_mjlab --python 3.11 -q --clear
cd /content/unitree_rl_mjlab
# mjlab 1.2.0（2026年3月公開）は MuJoCo 系の版を固定していないため、そのままだと最新版が入り
# 「mujoco と mujoco-warp の版が合わない」エラーになる。公開当時の組み合わせに固定する。
cat > /content/mjlab_pins.txt <<'EOF'
mujoco==3.5.0
mujoco-warp==3.5.0.2
warp-lang==1.12.0
EOF
uv pip install --python /content/venv_mjlab/bin/python -q --override /content/mjlab_pins.txt -e .     # mjlab==1.2.0 が入る
uv pip install --python /content/venv_mjlab/bin/python -q --override /content/mjlab_pins.txt matplotlib scipy   # scipy は mjlab の地形モジュールが使う（依存に書かれていない）
# 学習・変換・評価で使う部品をここでまとめて読み込んで確認（足りなければこのセルで止まる）
/content/venv_mjlab/bin/python - <<'EOF'
import sys, os; sys.path.insert(0, os.getcwd())
import torch, mjlab, mujoco, mujoco_warp, warp, scipy, importlib.metadata as m
import mjlab.entity, mjlab.scene, mjlab.envs, mjlab.rl, mjlab.tasks, src.tasks
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| mjlab', m.version('mjlab'),
      '| mujoco', mujoco.__version__, '| mujoco-warp', m.version('mujoco-warp'), '| warp', warp.__version__)
print('[ok] 必要な部品はすべて読み込めました')
EOF
# 学習ログはローカルに置き、ドライブへは監視セルが同期する（ドライブにあれば戻す）
mkdir -p logs/rsl_rl $P/logs/rsl_rl
bash $P/scripts/sync_logs.sh $P/logs/rsl_rl logs/rsl_rl
ls logs/rsl_rl/*/ 2>/dev/null | tail -5 || true

## 3. お手本を作り直す（`--contact_h 0.25`）

全編の加工前データ（`the_feels_full_raw.csv`）から、接地補正だけ条件を変えて作り直します。それ以外（推定ミスの補間、フィルタ）は元と同じ既定値です。
接地の計算とプレビュー動画には、元のお手本と同じ GMR の G1 モデルを使います。

In [ ]:
%%bash
set -e
PY=/content/venv_mjlab/bin/python
uv pip install --python $PY -q imageio imageio-ffmpeg     # プレビュー動画の書き出し用
bash $P/scripts/pin_clone.sh $URL_GMR /content/GMR $REV_GMR  # G1 のモデル（xml）だけ使う
XML=/content/GMR/assets/unitree_g1/g1_mocap_29dof.xml
ls $P/scripts/train_sim.py $P/scripts/render_policy.py
grep -q "action-rate-weight" $P/scripts/train_sim.py || { echo "[stop] train_sim.py が新しい版ではありません。手元の版をドライブに上書きしてください"; exit 1; }
grep -q "mean_action_rate" $P/scripts/render_policy.py || { echo "[stop] render_policy.py が新しい版ではありません。手元の版をドライブに上書きしてください"; exit 1; }
cd /content
MUJOCO_GL=egl $PY $P/scripts/gvhmr_to_g1_csv.py --from_csv $P/outputs/motions/${BASE}_raw.csv \
  --out_dir $P/outputs/motions --name $NAME --robot_xml $XML --contact_h 0.25
ls -la $P/outputs/motions/ | grep $NAME

## 4. 物理チェックで浮きが消えたか確認

元のお手本と同じ条件（mjlab の G1・摩擦 0.6）でチェックし、10 秒ごとの「お手本で低いほうの足の高さ」と「hover（浮いたまま上向きの力が必要）」を新旧で比べます。
**合格の目安**：110〜140 秒の足の高さが 0〜3cm 程度に下がり、hover が大きく減ること。

In [ ]:
%%bash
set -e
PY=/content/venv_mjlab/bin/python
XML=$($PY -c "from mjlab.asset_zoo.robots.unitree_g1.g1_constants import G1_XML; print(G1_XML)")
mkdir -p $P/outputs/check
$PY $P/scripts/check_reference.py --csv $P/outputs/motions/$NAME.csv --robot_xml $XML \
  --out_prefix $P/outputs/check/$NAME > $P/outputs/check/${NAME}_log.txt
$PY - <<EOF
import json, numpy as np
def load(n):
    c = np.genfromtxt(f"$P/outputs/check/{n}_check_frames.csv", delimiter=",", names=True)
    return c["time_s"], np.minimum(c["left_foot_h"], c["right_foot_h"]), c["hover"]
t0, f0, h0 = load("$BASE"); t1, f1, h1 = load("$NAME")
print(f"{'区間[秒]':>10} {'足の高さ 旧[cm]':>14} {'新[cm]':>8} {'hover 旧':>9} {'新':>5}")
for s in range(0, 140, 10):
    m0, m1 = (t0 >= s) & (t0 < s + 10), (t1 >= s) & (t1 < s + 10)
    print(f"{s:>4}〜{s+10:<4} {np.median(f0[m0])*100:>14.1f} {np.median(f1[m1])*100:>8.1f} {int(h0[m0].sum()):>9} {int(h1[m1].sum()):>5}")
for n in ["$BASE", "$NAME"]:
    j = json.load(open(f"$P/outputs/check/{n}_check.json"))
    print(n, j["frames_flagged"], "| 足が浮いている割合", j["airborne_ratio"])
r = json.load(open("$P/outputs/motions/${NAME}_prep_report.json"))
print("接地補正:", r["ground"], "| 接地中の足滑り 平均", r["after_ground"]["foot_slide_mean_mps"], "m/s")
EOF

In [ ]:
# 新旧のお手本のプレビュー（中央の画面になる動画）。終盤（110 秒以降）で影と足が離れていないかを見る
display(show(f"{P}/outputs/motions/{NAME}_retarget_preview.mp4", width=640))

## 5. 学習用の形式（npz）に変換

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
cp $P/outputs/motions/$NAME.csv src/assets/motions/g1/
/content/venv_mjlab/bin/python scripts/csv_to_npz.py --robot g1 \
  --input-file src/assets/motions/g1/$NAME.csv --output-name $NAME \
  --input-fps 30 --output-fps 50
cp src/assets/motions/g1/$NAME.npz $P/outputs/motions/
ls -la src/assets/motions/g1/ | grep the_feels_full

## 6. 追加学習（`model_19999` から 4,000 反復）

`--agent.resume True` で学習済みの重みを読み込み、**新しいお手本**と**キレ重視の報酬**で続きを学習します。新しい run 名は `the_feels_full_g25_ft` です（元の run は上書きされません）。
ログの反復番号は 19999 の続き（〜23999）になります。ログ先頭の `[train_sim]` 行で、報酬の設定が反映されていることを確認してください。
学習を始めたら、**すぐ下の監視セルを実行したまま**にします。

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
if [ -f /content/train.pid ] && [ -d /proc/$(cat /content/train.pid) ]; then
  echo "[stop] 学習がすでに動いています（PID $(cat /content/train.pid)）。"; exit 1
fi
[ -f logs/rsl_rl/g1_tracking/$BASE_RUN/$BASE_CKPT ] || { echo "[stop] 読み込むチェックポイントがありません: $BASE_RUN/$BASE_CKPT"; exit 1; }
mkdir -p $P/logs
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz \
  --env.scene.num-envs 4096 \
  --agent.max-iterations $ITERS \
  --agent.logger tensorboard \
  --agent.run-name ${NAME}_ft \
  --agent.resume True --agent.load-run "$BASE_RUN" --agent.load-checkpoint "$BASE_CKPT" \
  --push-scale 0.5 \
  --action-rate-weight $ACTION_RATE --body-pos-std $BODY_POS_STD --body-ori-std $BODY_ORI_STD \
  > $P/logs/train_$NAME.log 2>&1 &
echo $! > /content/train.pid
echo $ITERS > /content/train.iters
sleep 90; grep -E "train_sim|Loading model|Error|error" $P/logs/train_$NAME.log | head
echo "学習を開始しました（PID $(cat /content/train.pid)）。ログ: $P/logs/train_$NAME.log"

In [ ]:
# 監視セル：学習が終わるまで実行したままにする（止めても学習は止まらない）
import re, subprocess, time
from IPython.display import clear_output
pid = int(open('/content/train.pid').read())
log = f"{P}/logs/train_{NAME}.log"
alive = lambda: os.path.exists(f"/proc/{pid}")
sync = lambda: subprocess.run(['bash', f'{P}/scripts/sync_logs.sh', f'{L}/logs/rsl_rl', f'{P}/logs/rsl_rl'])
last = time.time()
while alive():
    time.sleep(60)
    txt = open(log, errors='ignore').read().splitlines()
    keep = [l.strip() for l in txt if re.search(r'Learning iteration|Mean episode length|Mean reward|ETA|Error|error', l)]
    clear_output(wait=True)
    print(time.strftime('%H:%M:%S'), f'学習中 (PID {pid})'); print('\n'.join(keep[-5:]))
    if time.time() - last > 600:
        sync(); last = time.time(); print('[sync] ドライブへ同期しました')
sync()
txt = open(log, errors='ignore').read()
print('学習プロセスが終了しました。ログの最後:'); print('\n'.join(txt.splitlines()[-15:]))

# 学習が正常に終わったかを確認（失敗していたら、ここで止めて後ろのセルを実行させない）
import glob
runs = sorted(glob.glob(f"{L}/logs/rsl_rl/g1_tracking/*_{NAME}*"), key=os.path.getmtime)
ckpts = glob.glob(f"{runs[-1]}/model_*.pt") if runs else []
its = [int(m) for m in re.findall(r'Learning iteration\s+(\d+)\s*/', txt)]
want = int(open('/content/train.iters').read()) if os.path.exists('/content/train.iters') else None
print(f"[check] run: {runs[-1] if runs else 'なし'} / チェックポイント {len(ckpts)} 個 / 最後の反復 {max(its) if its else 'なし'} / 予定 {want}")
if 'Traceback' in txt or not ckpts:
    raise RuntimeError('学習が途中でエラー終了しました。上のログの最後を確認してください（後ろのセルは実行しません）。')
if want and its and max(its) < want * 0.95:
    raise RuntimeError(f'学習が予定の反復数まで進まずに終わりました（{max(its)}/{want}）。ログを確認してください。')
print('[ok] 学習は正常に終わりました。評価と動画の書き出しに進みます。')

### 切断されたとき

1・2・5 を実行し直してから、次のセルの `REMAIN` に残りの反復回数を入れて実行し、監視セルをもう一度実行します（直近の `_ft` の run から続きを学習します）。一括実行では動きません。

In [ ]:
%%bash
REMAIN=0      # ← 残りの反復回数を入れたときだけ実行される
if [ "$REMAIN" = "0" ]; then echo "[skip] 再開しません（REMAIN=0）"; exit 0; fi
cd /content/unitree_rl_mjlab
RUN=$(basename $(ls -td logs/rsl_rl/g1_tracking/*_${NAME}_ft* | head -1))
echo "再開する run: $RUN / $(ls -v logs/rsl_rl/g1_tracking/$RUN/model_*.pt | tail -1)"
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz --env.scene.num-envs 4096 \
  --agent.max-iterations $REMAIN --agent.logger tensorboard --agent.run-name ${NAME}_ft_resume \
  --agent.resume True --agent.load-run "$RUN" --push-scale 0.5 \
  --action-rate-weight $ACTION_RATE --body-pos-std $BODY_POS_STD --body-ori-std $BODY_ORI_STD \
  > $P/logs/train_${NAME}_resume.log 2>&1 &
echo $! > /content/train.pid; echo $REMAIN > /content/train.iters; echo started

## 7. 評価（摩擦 0.8 は動画つき、0.5・1.0 は評価値だけ）

ノートブック2の 3-6 と同じ評価に加えて、**元の方策（`model_19999`）を新しいお手本で評価**した値も出します。
「お手本を直しただけの効果」と「追加学習の効果」を分けて示せるので、レポートの比較表にそのまま使えます。
`act変化` は 1 ステップ（0.02 秒）ごとの方策の出力の変化の平均で、追加学習で大きくなりすぎていれば震えのサインです。

In [ ]:
%%bash
set -eo pipefail
cd /content/unitree_rl_mjlab
RUN=$(ls -td logs/rsl_rl/g1_tracking/*_${NAME}* | sed -n 1p)   # 一番新しい run（再開した run も含む）
[ -n "$RUN" ] || { echo "[stop] 学習の run が見つかりません"; exit 1; }
CKPT=$(ls -v $RUN/model_*.pt | tail -1)
TAG=$(basename $CKPT .pt)
[ -n "$CKPT" ] || { echo "[stop] チェックポイントが見つかりません（学習が終わっていない可能性）"; exit 1; }
echo "評価するチェックポイント: $CKPT"
mkdir -p $P/outputs/videos $P/outputs/eval
PY=/content/venv_mjlab/bin/python
$PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
  --checkpoint $CKPT --friction 0.8 --out $P/outputs/videos/policy_${NAME}_${TAG}.mp4 | tail -3
# 途中で転倒して動画が短い（または作られない）ときは、終了条件を外して全編の動画も書き出す（見せる用）
if python3 -c "import json,sys; sys.exit(0 if json.load(open('$P/outputs/videos/policy_${NAME}_${TAG}.json'))['success'] else 1)"; then
  echo "[ok] 転倒なしで最後まで踊れました"
else
  echo "[info] 転倒したため、終了条件を外した全編の動画も書き出します（_noterm.mp4）"
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
    --checkpoint $CKPT --friction 0.8 --no-terminations --out $P/outputs/videos/policy_${NAME}_${TAG}_noterm.mp4 | grep -E "^\[" || true
fi
for F in 0.5 1.0; do
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
    --checkpoint $CKPT --friction $F --no-video --out $P/outputs/eval/policy_${NAME}_${TAG}_mu$F.mp4 > /dev/null
done
python3 - <<EOF
import json, glob
fs = ["$P/outputs/videos/policy_${NAME}_${TAG}.json"] + sorted(glob.glob("$P/outputs/eval/policy_${NAME}_${TAG}_mu*.json"))
print(f"{'摩擦':>5} {'成功':>5} {'転倒時刻[s]':>10} {'関節誤差[rad]':>12} {'位置誤差[m]':>10}  止まった条件")
for f in fs:
    j = json.load(open(f))
    print(f"{j['friction']:>5} {str(j['success']):>5} {str(j['first_fall_s']):>10} {j['mean_joint_err_rad']:>12} {j['mean_body_pos_err_rel_m']:>10}  {j.get('fall_reason') or ''}")
import os
nt = "$P/outputs/videos/policy_${NAME}_${TAG}_noterm.json"
if os.path.exists(nt):
    j = json.load(open(nt))
    print(f"[終了条件なし・全編] 長さ {j['duration_s']} 秒 / 関節誤差 {j['mean_joint_err_rad']} rad / 位置誤差 平均 {j['mean_body_pos_err_rel_m']} m・最大 {j['max_body_pos_err_rel_m']} m")
EOF

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
PY=/content/venv_mjlab/bin/python
NEW=$(ls -td logs/rsl_rl/g1_tracking/*_${NAME}_ft* | sed -n 1p); NEWCK=$(ls -v $NEW/model_*.pt | tail -1)
OLDCK=logs/rsl_rl/g1_tracking/$BASE_RUN/$BASE_CKPT
# 旧お手本の npz（ノートブック2の 3-3 でドライブに保存済み。なければ作る）
if [ -f $P/outputs/motions/$BASE.npz ]; then cp $P/outputs/motions/$BASE.npz src/assets/motions/g1/
else cp $P/outputs/motions/$BASE.csv src/assets/motions/g1/
  $PY scripts/csv_to_npz.py --robot g1 --input-file src/assets/motions/g1/$BASE.csv --output-name $BASE --input-fps 30 --output-fps 50; fi
mkdir -p $P/outputs/eval
run() {  # $1=checkpoint $2=npz の名前 $3=出力名
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$2.npz \
    --checkpoint $1 --friction 0.8 --no-video --out $P/outputs/eval/$3.mp4 > /dev/null
}
run $OLDCK $BASE  cmp_old_policy_old_ref
run $OLDCK $NAME  cmp_old_policy_new_ref
run $NEWCK $NAME  cmp_new_policy_new_ref
$PY - <<EOF
import json
rows = [("旧方策 × 旧お手本（ノートブック2）", "cmp_old_policy_old_ref"),
        ("旧方策 × 新お手本（お手本の修正だけ）", "cmp_old_policy_new_ref"),
        ("新方策 × 新お手本（修正＋追加学習）", "cmp_new_policy_new_ref")]
print(f"{'条件':<26} {'成功':>5} {'関節誤差':>8} {'位置誤差':>8} {'最大':>7} {'最大ドリフト':>10} {'act変化':>8}")
for label, n in rows:
    j = json.load(open(f"$P/outputs/eval/{n}.json"))
    print(f"{label:<26} {str(j['success']):>5} {j['mean_joint_err_rad']:>8} {j['mean_body_pos_err_rel_m']:>8} "
          f"{j['max_body_pos_err_rel_m']:>7} {j['max_root_xy_drift_m']:>10} {j['mean_action_rate']:>8}")
EOF

In [ ]:
import glob
vids = sorted(glob.glob(f"{P}/outputs/videos/policy_{NAME}_model_*.mp4"), key=os.path.getmtime)
if not vids:
    print("動画がありません（評価で開始直後に止まった可能性）。3-6 の出力を確認してください。")
else:
    latest = vids[-1]
    print(latest, f"({os.path.getsize(latest) / 1e6:.1f} MB)")
    display(show(latest))

## 8. 比較動画を作る

左：人のメッシュ／中央：**新しいお手本**（`the_feels_full_g25_retarget_preview.mp4`）／右：追加学習した G1。出力は `comparison_the_feels_full_g25.mp4` です（元の比較動画は上書きされません）。

In [ ]:
%%bash
set -e
POLICY=$(ls -t $P/outputs/videos/policy_${NAME}_model_*.mp4 | head -1)   # 採用する動画に書き換えてよい
echo "採用: $POLICY  (START=$START)"
START=$START bash $P/scripts/make_comparison.sh \
  $P/outputs/gvhmr/the_feels_full/2_global.mp4 \
  $P/outputs/motions/${NAME}_retarget_preview.mp4 \
  "$POLICY" \
  $P/input/the_feels_full.mp4 \
  $P/outputs/videos/comparison_$NAME.mp4

In [ ]:
show(f"{P}/outputs/videos/comparison_{NAME}.mp4", width=1100)